# LLM Benchmark – Interview AI Evaluation

Notebook ini membandingkan dua model LLM melalui OpenRouter:
- **DeepSeek V4 Flash** (`deepseek/deepseek-v4-flash`)
- **Gemini 2.5 Flash** (`google/gemini-2.5-flash`)

Dua tugas dievaluasi:
1. **Question Generation** – menghasilkan pertanyaan wawancara berdasarkan deskripsi pekerjaan dan ringkasan CV.
2. **Answer Evaluation** – memberikan umpan balik terstruktur terhadap jawaban kandidat dalam wawancara.

Setiap tugas dinilai menggunakan **DeepEval GEval** dengan `openai/gpt-4o-mini` sebagai model juri, diakses melalui OpenRouter.

## 0. Dependency Installation

Sel ini memeriksa dan menginstal paket-paket yang dibutuhkan.

In [2]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "deepeval": "deepeval",
    "openai": "openai",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "kagglehub": "kagglehub",
}

for module, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module)
        print(f" {pip_name} already installed")
    except ImportError:
        print(f" Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])
        print(f" {pip_name} installed successfully")

 deepeval already installed
 openai already installed
 pandas already installed
 matplotlib already installed
 seaborn already installed
 kagglehub already installed


## 1. Imports & Configuration

Inisialisasi klien OpenAI yang mengarah ke OpenRouter, dan konfigurasi `deepeval` agar menggunakan model juri yang sama.

In [3]:
import os
import json
import time
import textwrap
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from openai import OpenAI

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# ── OpenRouter client ─────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY environment variable is not set. "
        "Please export it before running this notebook."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# ── Models under test ─────────────────────────────────────────────────────────
MODELS = [
    "deepseek/deepseek-v4-flash",
    "google/gemini-2.5-flash",
]

JUDGE_MODEL = "openai/gpt-4o-mini"   # accessed via OpenRouter

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.dirname(os.path.abspath("__file__"))
QGEN_CSV   = os.path.join(OUTPUT_DIR, "question_gen_results.csv")
EVAL_CSV   = os.path.join(OUTPUT_DIR, "answer_eval_results.csv")

print("Configuration loaded successfully.")
print(f"Models under test : {MODELS}")
print(f"Judge model       : {JUDGE_MODEL}")
print(f"Results directory : {OUTPUT_DIR}")

Configuration loaded successfully.
Models under test : ['deepseek/deepseek-v4-flash', 'google/gemini-2.5-flash']
Judge model       : openai/gpt-4o-mini
Results directory : /home/fadawkas/Documents/FinalArc/vagmiAI/ai_benchmark/LLM


## 0a. Load Real Dataset from Kaggle

Download and prepare the AI Recruitment Pipeline Dataset.

In [4]:
import kagglehub
import pandas as pd
import re

# Download latest version of the dataset
download_path = kagglehub.dataset_download("yaswanthkumary/ai-recruitment-pipeline-dataset")
print("Dataset downloaded to:", download_path)

# The actual CSV file inside the downloaded folder
csv_file = f"{download_path}/dataset.csv"
df = pd.read_csv(csv_file)
print(f"Total records: {len(df)}")

# Keep the first 100 records (or random_sample for reproducibility)
df = df.head(100).copy()
print(f"Using {len(df)} records for benchmarking.")

Dataset downloaded to: /home/fadawkas/.cache/kagglehub/datasets/yaswanthkumary/ai-recruitment-pipeline-dataset/versions/1
Total records: 10174
Using 100 records for benchmarking.


### Preprocessing for Answer Evaluation

Each transcript is a multi‑turn dialogue. We extract the longest candidate utterance and the question that immediately precedes it.

In [5]:
def extract_longest_answer_and_question(transcript: str):
    """Parse transcript and return (question, answer)."""
    turns = re.split(r'(Interviewer:|[A-Z][a-zA-Z]+ [A-Z][a-zA-Z]+:)', transcript)
    speakers = []
    texts = []
    for i in range(1, len(turns), 2):
        speakers.append(turns[i].strip().rstrip(':'))
        texts.append(turns[i+1].strip() if i+1 < len(turns) else '')

    candidate_turns = [(speaker, text) for speaker, text in zip(speakers, texts) if speaker != "Interviewer"]
    if not candidate_turns:
        return "", ""

    longest_idx, longest_len = max(enumerate(candidate_turns), key=lambda x: len(x[1][1].split()))
    longest_answer = candidate_turns[longest_idx][1]

    original_idx = 0
    found = False
    for idx, (spk, txt) in enumerate(zip(speakers, texts)):
        if spk != "Interviewer" and txt == longest_answer:
            original_idx = idx
            found = True
            break
    if not found:
        return "", longest_answer
    if original_idx > 0 and speakers[original_idx-1] == "Interviewer":
        question = texts[original_idx-1]
        return question, longest_answer
    else:
        return "", longest_answer

df[['extracted_question', 'extracted_answer']] = df['Transcript'].apply(
    lambda x: pd.Series(extract_longest_answer_and_question(x))
)

df_clean = df[(df['extracted_question'] != '') & (df['extracted_answer'] != '')].copy()
print(f"Records after extraction: {len(df_clean)}")

if len(df_clean) < 10:
    print("Warning: very few answer evaluation cases extracted. Consider using synthetic ones instead.")
else:
    answer_eval_df = df_clean.head(100).copy()

Records after extraction: 97


## 2. Task 1 – Interview Question Generation

### 2.1 Test Cases

Sepuluh kasus uji realistis dalam bahasa Indonesia, mencakup berbagai industri dan level senioritas.

In [10]:
QUESTION_GEN_CASES = []
for i, row in df.head(100).iterrows():
    QUESTION_GEN_CASES.append({
        "id": f"QG-{i+1:03d}",
        "job_description": f"Role: {row['Role']}\nJob Description: {row['Job_Description']}",
        "cv_summary": "Summary from resume: " + (row['Resume'][:300] + "..." if isinstance(row['Resume'], str) and len(row['Resume']) > 300 else str(row['Resume']))
    })
print(f"Loaded {len(QUESTION_GEN_CASES)} question generation test cases (real data).")

Loaded 100 question generation test cases (real data).


### 2.2 Prompt Builder – Question Generation

In [11]:
def build_qgen_messages(case: dict) -> list:
    """Build chat messages for the interview question generation task."""
    system = (
        "Anda adalah seorang HR profesional berpengalaman yang ahli dalam menyusun "
        "pertanyaan wawancara kerja. Tugas Anda adalah menghasilkan daftar pertanyaan "
        "wawancara yang relevan, mendalam, dan terstruktur berdasarkan deskripsi pekerjaan "
        "dan ringkasan CV kandidat yang diberikan. Gunakan bahasa Indonesia yang profesional."
    )
    user = (
        f"**Deskripsi Pekerjaan:**\n{case['job_description']}\n\n"
        f"**Ringkasan CV Kandidat:**\n{case['cv_summary']}\n\n"
        "Buatlah **8 pertanyaan wawancara** yang mencakup:\n"
        "- Kompetensi teknis yang relevan dengan posisi\n"
        "- Pengalaman kerja dan pencapaian kandidat\n"
        "- Soft skills dan kemampuan kepemimpinan (jika relevan)\n"
        "- Situational/behavioral questions\n"
        "- Potensi gap antara profil kandidat dan kebutuhan posisi\n\n"
        "Format output: nomor urut diikuti pertanyaan yang jelas dan spesifik."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def call_model(model: str, messages: list, max_retries: int = 3) -> str:
    """Call a model via OpenRouter and return the text response."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.7,
                max_tokens=2048,
            )
            return resp.choices[0].message.content.strip()
        except Exception as exc:
            print(f"  Attempt {attempt}/{max_retries} failed for '{model}': {exc}")
            if attempt < max_retries:
                time.sleep(2 ** attempt)   # exponential back-off
    return "[ERROR: model call failed after retries]"

print("Helpers defined.")

Helpers defined.


### 2.3 GEval Metric – Question Generation

Metrik ini menilai kualitas pertanyaan yang dihasilkan berdasarkan: relevansi, cakupan, kejelasan, dan kedalaman.

In [12]:
import deepeval

# Override the LLM used by deepeval to our OpenRouter-hosted GPT-4o-mini
# deepeval supports setting a custom model; we wrap it via the openai compatibility layer.
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

QGEN_CRITERIA = textwrap.dedent("""
    Anda mengevaluasi kualitas daftar pertanyaan wawancara yang dihasilkan oleh AI.
    Berikan skor dari 0 hingga 1 berdasarkan empat dimensi berikut:

    1. **Relevansi** (0.25): Apakah setiap pertanyaan relevan dengan posisi dan profil kandidat?
    2. **Cakupan** (0.25): Apakah pertanyaan mencakup kompetensi teknis, pengalaman, perilaku, dan potensi gap?
    3. **Kejelasan** (0.25): Apakah pertanyaan dirumuskan dengan jelas, spesifik, dan mudah dipahami?
    4. **Kedalaman** (0.25): Apakah pertanyaan mendorong jawaban yang substantif dan bukan sekadar yes/no?

    Keluarkan respons Anda HANYA dalam format JSON berikut (tanpa markdown fence):
    {"score": <float 0-1>, "reason": "<penjelasan singkat dalam 2-3 kalimat>"}
""")

QGEN_STEPS = [
    "Baca deskripsi pekerjaan dan ringkasan CV di bagian input.",
    "Baca daftar pertanyaan wawancara yang dihasilkan (actual_output).",
    "Nilai relevansi setiap pertanyaan terhadap posisi dan profil kandidat.",
    "Periksa apakah pertanyaan mencakup aspek teknis, pengalaman, perilaku, dan gap.",
    "Nilai kejelasan dan spesifisitas setiap pertanyaan.",
    "Nilai apakah pertanyaan dirancang untuk mendapatkan jawaban mendalam.",
    "Hitung skor rata-rata dari keempat dimensi (masing-masing bernilai 0.25).",
    "Keluarkan JSON dengan kunci 'score' (float 0-1) dan 'reason' (string penjelasan).",
]

qgen_metric = GEval(
    name="InterviewQuestionQuality",
    criteria=QGEN_CRITERIA,
    evaluation_steps=QGEN_STEPS,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model=JUDGE_MODEL,
    threshold=0.5,
)

print("GEval metric for Question Generation created.")

GEval metric for Question Generation created.


### 2.4 Run Benchmark – Question Generation

Memanggil setiap model untuk setiap kasus uji dan mengukur hasilnya dengan metrik GEval.

In [ ]:
def run_geval(metric: GEval, input_text: str, output_text: str) -> tuple[float, str]:
    """Run a GEval metric and return (score, reason)."""
    test_case = LLMTestCase(input=input_text, actual_output=output_text)
    metric.measure(test_case)
    score = metric.score if metric.score is not None else 0.0
    reason = metric.reason if metric.reason else ""
    return round(float(score), 4), reason


qgen_records = []

for model in MODELS:
    model_label = model.split("/")[-1]
    print(f"\nModel: {model_label}")

    for case in QUESTION_GEN_CASES:
        print(f"  → Case {case['id']} ", end="", flush=True)

        # Build input string for GEval context
        input_text = (
            f"Deskripsi Pekerjaan:\n{case['job_description']}\n\n"
            f"Ringkasan CV:\n{case['cv_summary']}"
        )

        # Call the model
        messages = build_qgen_messages(case)
        output_text = call_model(model, messages)
        print("[response OK] ", end="", flush=True)

        # Evaluate with GEval
        try:
            score, reason = run_geval(qgen_metric, input_text, output_text)
        except Exception as exc:
            print(f"[GEval error: {exc}]")
            score, reason = 0.0, str(exc)

        print(f"score={score:.3f}")

        qgen_records.append({
            "model": model_label,
            "case_id": case["id"],
            "score": score,
            "reason": reason,
            "output": output_text,
        })

        time.sleep(1)   # small pause between calls

df_qgen = pd.DataFrame(qgen_records)
df_qgen.to_csv(QGEN_CSV, index=False)
print(f"\nQuestion generation results saved to: {QGEN_CSV}")
df_qgen[["model", "case_id", "score"]].head(10)

## 3. Task 2 – Interview Answer Evaluation

### 3.1 Test Cases

Sepuluh skenario evaluasi jawaban wawancara dalam bahasa Indonesia.

In [13]:
ANSWER_EVAL_CASES = []
for i, row in answer_eval_df.head(100).iterrows():
    ANSWER_EVAL_CASES.append({
        "id": f"AE-{i+1:03d}",
        "question": row['extracted_question'],
        "transcript": row['extracted_answer'],
        "nonverbal_summary": "No nonverbal cues available (dataset text only)."
    })
print(f"Loaded {len(ANSWER_EVAL_CASES)} answer evaluation test cases (real data).")

Loaded 97 answer evaluation test cases (real data).


### 3.2 Prompt Builder – Answer Evaluation

In [14]:
def build_answer_eval_messages(case: dict) -> list:
    """Build chat messages for the interview answer evaluation task."""
    system = (
        "Anda adalah seorang evaluator wawancara kerja berpengalaman. Tugas Anda adalah "
        "menganalisis jawaban kandidat dalam wawancara dan memberikan umpan balik yang "
        "terstruktur, konstruktif, dan actionable. Gunakan bahasa Indonesia yang profesional."
    )
    user = (
        f"**Pertanyaan Wawancara:**\n{case['question']}\n\n"
        f"**Transkrip Jawaban Kandidat:**\n{case['transcript']}\n\n"
        f"**Ringkasan Isyarat Non-Verbal:**\n{case['nonverbal_summary']}\n\n"
        "Berikan evaluasi terstruktur dengan format berikut:\n"
        "**Skor Keseluruhan:** [X dari 10]\n"
        "**Kekuatan:** [2–3 poin utama yang dilakukan dengan baik]\n"
        "**Area Perbaikan:** [2–3 poin yang perlu ditingkatkan]\n"
        "**Rekomendasi Spesifik:** [saran konkret yang dapat langsung diterapkan kandidat]\n"
        "**Kesimpulan:** [1–2 kalimat penutup evaluasi]"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

print("Answer evaluation prompt builder defined.")

Answer evaluation prompt builder defined.


### 3.3 GEval Metric – Answer Evaluation

Metrik ini menilai kualitas umpan balik evaluasi jawaban berdasarkan: akurasi feedback, actionability, kejelasan, dan nada konstruktif.

In [15]:
ANSWER_EVAL_CRITERIA = textwrap.dedent("""
    Anda mengevaluasi kualitas umpan balik yang diberikan AI terhadap jawaban kandidat wawancara.
    Berikan skor dari 0 hingga 1 berdasarkan empat dimensi berikut:

    1. **Akurasi Umpan Balik** (0.25): Apakah poin kekuatan dan kelemahan yang diidentifikasi akurat 
       dan mencerminkan isi jawaban serta isyarat non-verbal dengan tepat?
    2. **Actionability** (0.25): Apakah rekomendasi yang diberikan spesifik, praktis, dan dapat 
       langsung diterapkan oleh kandidat?
    3. **Kejelasan** (0.25): Apakah umpan balik disampaikan dengan jelas, terstruktur, dan mudah 
       dipahami?
    4. **Nada Konstruktif** (0.25): Apakah umpan balik bersifat membangun, seimbang, dan tidak 
       bersifat menghakimi secara berlebihan?

    Keluarkan respons Anda HANYA dalam format JSON berikut (tanpa markdown fence):
    {"score": <float 0-1>, "reason": "<penjelasan singkat dalam 2-3 kalimat>"}
""")

ANSWER_EVAL_STEPS = [
    "Baca pertanyaan wawancara, transkrip jawaban, dan ringkasan non-verbal di bagian input.",
    "Baca umpan balik evaluasi yang dihasilkan AI (actual_output).",
    "Periksa apakah poin kekuatan dan kelemahan benar-benar tercermin dari jawaban kandidat.",
    "Nilai apakah rekomendasi bersifat spesifik dan dapat segera diterapkan.",
    "Nilai kejelasan struktur dan keterbacaan umpan balik.",
    "Nilai apakah nada umpan balik konstruktif, seimbang, dan tidak menghakimi.",
    "Hitung skor rata-rata dari keempat dimensi (masing-masing bernilai 0.25).",
    "Keluarkan JSON dengan kunci 'score' (float 0-1) dan 'reason' (string penjelasan).",
]

answer_eval_metric = GEval(
    name="InterviewFeedbackQuality",
    criteria=ANSWER_EVAL_CRITERIA,
    evaluation_steps=ANSWER_EVAL_STEPS,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model=JUDGE_MODEL,
    threshold=0.5,
)

print("GEval metric for Answer Evaluation created.")

GEval metric for Answer Evaluation created.


### 3.4 Run Benchmark – Answer Evaluation

In [ ]:
answer_eval_records = []

for model in MODELS:
    model_label = model.split("/")[-1]
    print(f"\nModel: {model_label}")

    for case in ANSWER_EVAL_CASES:
        print(f"  → Case {case['id']} ", end="", flush=True)

        input_text = (
            f"Pertanyaan: {case['question']}\n\n"
            f"Transkrip Jawaban: {case['transcript']}\n\n"
            f"Non-Verbal: {case['nonverbal_summary']}"
        )

        messages = build_answer_eval_messages(case)
        output_text = call_model(model, messages)
        print("[response OK] ", end="", flush=True)

        try:
            score, reason = run_geval(answer_eval_metric, input_text, output_text)
        except Exception as exc:
            print(f"[GEval error: {exc}]")
            score, reason = 0.0, str(exc)

        print(f"score={score:.3f}")

        answer_eval_records.append({
            "model": model_label,
            "case_id": case["id"],
            "score": score,
            "reason": reason,
            "output": output_text,
        })

        time.sleep(1)

df_eval = pd.DataFrame(answer_eval_records)
df_eval.to_csv(EVAL_CSV, index=False)
print(f"\nAnswer evaluation results saved to: {EVAL_CSV}")
df_eval[["model", "case_id", "score"]].head(10)

In [18]:
def run_geval(metric: GEval, input_text: str, output_text: str) -> tuple[float, str]:
    """Run a GEval metric and return (score, reason)."""
    test_case = LLMTestCase(input=input_text, actual_output=output_text)
    metric.measure(test_case)
    score = metric.score if metric.score is not None else 0.0
    reason = metric.reason if metric.reason else ""
    return round(float(score), 4), reason

## 4. Aggregate Statistics

Statistik deskriptif per model untuk setiap tugas.

In [ ]:
def print_stats(df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    print(f"\n{'═'*60}")
    print(f"  TASK: {task_name}")
    print(f"{'═'*60}")
    stats = (
        df.groupby("model")["score"]
        .agg(["mean", "std", "min", "max", "count"])
        .round(4)
        .rename(columns={"mean": "Mean", "std": "Std", "min": "Min", "max": "Max", "count": "N"})
    )
    print(stats.to_string())
    return stats

stats_qgen = print_stats(df_qgen, "Interview Question Generation")
stats_eval = print_stats(df_eval, "Interview Answer Evaluation")


════════════════════════════════════════════════════════════
  TASK: Interview Question Generation
════════════════════════════════════════════════════════════
                    Mean     Std  Min  Max    N
model                                          
deepseek-v4-flash  0.896  0.0197  0.8  0.9  100
gemini-2.5-flash   0.890  0.0438  0.6  1.0  100

════════════════════════════════════════════════════════════
  TASK: Interview Answer Evaluation
════════════════════════════════════════════════════════════
                     Mean     Std  Min  Max   N
model                                          
deepseek-v4-flash  0.7031  0.0620  0.5  0.8  97
gemini-2.5-flash   0.7402  0.0533  0.6  0.8  97


## 5. Ringkasan Hasil Benchmark

Tabel ringkasan statistik dan interpretasi awal dari hasil benchmark.

In [ ]:
def summarize(df_qg: pd.DataFrame, df_ae: pd.DataFrame):
    rows = []
    for model_raw in MODELS:
        label = model_raw.split("/")[-1]
        qg_scores = df_qg.loc[df_qg["model"] == label, "score"]
        ae_scores = df_ae.loc[df_ae["model"] == label, "score"]
        rows.append({
            "Model": label,
            "QGen Mean": round(qg_scores.mean(), 4),
            "QGen Std": round(qg_scores.std(), 4),
            "AnsEval Mean": round(ae_scores.mean(), 4),
            "AnsEval Std": round(ae_scores.std(), 4),
            "Overall Mean": round(
                pd.concat([qg_scores, ae_scores]).mean(), 4
            ),
        })
    return pd.DataFrame(rows).set_index("Model")

summary_df = summarize(df_qgen, df_eval)
print("\nBenchmark Summary Table")
print("=" * 70)
print(summary_df.to_string())
print("=" * 70)

best_qgen = summary_df["QGen Mean"].idxmax()
best_eval = summary_df["AnsEval Mean"].idxmax()
best_overall = summary_df["Overall Mean"].idxmax()

print(f"\nBest model – Question Generation : {best_qgen}")
print(f"Best model – Answer Evaluation   : {best_eval}")
print(f"Best model – Overall             : {best_overall}")


Benchmark Summary Table
                   QGen Mean  QGen Std  AnsEval Mean  AnsEval Std  Overall Mean
Model                                                                          
deepseek-v4-flash      0.896    0.0197        0.7031       0.0620        0.8010
gemini-2.5-flash       0.890    0.0438        0.7402       0.0533        0.8162

Best model – Question Generation : deepseek-v4-flash
Best model – Answer Evaluation   : gemini-2.5-flash
Best model – Overall             : gemini-2.5-flash
